In [1]:
import numpy as np
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from scipy.stats import norm

# =========================================================
# WEEK 13 — FUNCTION 7 (RL-AUGMENTED, PCA-GUIDED, CLUSTER-AWARE BO, STRICT [0,1])
#
# RL ideas applied:
# - Exploration–exploitation: epsilon-greedy selection (decays with more data).
# - Feedback-driven adaptation: "bandit" over candidate pools (global/local/centroid/boundary)
#   using optimistic EI statistics as pool value estimates.
# - "Self-play" analogy: the agent continually proposes challengers to the current best,
#   choosing between uncertainty-seeking moves and improvement-seeking moves.
#
# Output constraint:
# - Prints ONLY the recommended next x (<= 6 decimals).
# =========================================================

# -----------------------------
# 1) Data (Function 7)
# -----------------------------
X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245 , 1.024693, 1.02457 , 1.061017, 1.098654, 1.051013],
    [0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371],
    [0.611853, 0.139495, 0.292145, 0.366362, 0.45607 , 0.785175],
    [0.015006, 0.390905, 0.178469, 0.119929, 0.088415, 0.904408],
    [0.046821, 0.309546, 0.608802, 0.064364, 0.39334 , 0.990644],
    [0.011478, 0.62027 , 0.525606, 0.053535, 0.52488 , 0.666127],
    [0.028679, 0.235471, 0.148723, 0.076614, 0.11285 , 0.837107],
    [0.069198, 0.39455 , 0.352452, 0.093928, 0.370707, 0.725655],
    [0.000000, 0.367783, 0.346341, 0.052998, 0.363011, 0.730958],
    [0.000000, 0.319509, 0.283024, 0.207785, 0.339392, 0.737674],
    [0.000000, 0.330508, 0.317636, 0.248626, 0.321670, 0.716418],
    [0.000000, 0.364378, 0.290209, 0.285716, 0.309745, 0.697649]
], dtype=float)

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.636858051500375e-06, 1.680828424430851,
    1.1170576710554418, 0.44099891630237703, 0.8950628737420184, 0.664856997347448,
    0.6583383225997628, 1.6173276124769211, 1.3305832886811908, 2.1047631312317456,
    2.2979366498230593, 2.2166754301653624
], dtype=float)

# -----------------------------
# 2) Config (Week 13)
# -----------------------------
RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Candidate budgets
N_GLOBAL = 1600
N_LOCAL_BEST = 1200
N_LOCAL_CENTROID = 1000
N_BOUNDARY = 800

# PCA retention
PCA_KEEP_VAR = 0.95

# Min-distance filter
MIN_DIST = 7.5e-4

# -----------------------------
# 3) Acquisition helpers
# -----------------------------
def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-12)
    z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

def zscore(a):
    s = np.std(a)
    return (a - np.mean(a)) / s if s > 1e-12 else (a - np.mean(a))

# -----------------------------
# 4) Candidate generation
# -----------------------------
def sobol_global_candidates(n, d, seed):
    eng = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=seed)
    return eng.draw(n).numpy()

def min_dist_filter(Xcand, Xtrain, thr):
    if Xcand.shape[0] == 0:
        return Xcand
    thr2 = thr * thr
    dist2 = ((Xcand[:, None, :] - Xtrain[None, :, :]) ** 2).sum(axis=2)
    keep = dist2.min(axis=1) > thr2
    return Xcand[keep]

def pca_model(X01, seed, keep_var=0.95):
    xs = StandardScaler()
    Xs = xs.fit_transform(X01)

    p_full = PCA(random_state=seed)
    p_full.fit(Xs)
    cum = np.cumsum(p_full.explained_variance_ratio_)
    m = int(np.searchsorted(cum, keep_var) + 1)
    m = max(2, min(m, X01.shape[1]))

    pca = PCA(n_components=m, random_state=seed)
    Z = pca.fit_transform(Xs)
    return xs, pca, Z

def local_candidates_in_pc(center01, n, xs, pca, base_scale, seed):
    rng = np.random.RandomState(seed)
    center_s = xs.transform(center01.reshape(1, -1))
    center_z = pca.transform(center_s).ravel()

    eig = pca.explained_variance_
    rel = np.sqrt(eig / np.max(eig))  # bigger steps along top PCs
    scales = base_scale * rel

    Z = center_z + rng.normal(0.0, scales, size=(n, center_z.size))
    Xs_rec = pca.inverse_transform(Z)
    X01 = xs.inverse_transform(Xs_rec)
    return np.clip(X01, 0.0, 1.0)

def boundary_candidates_in_pc(centroid_top01, centroids_other01, n, xs, pca, jitter, seed):
    rng = np.random.RandomState(seed)
    if centroids_other01.shape[0] == 0:
        return np.empty((0, centroid_top01.size))

    top_z = pca.transform(xs.transform(centroid_top01.reshape(1, -1))).ravel()
    other_z = pca.transform(xs.transform(centroids_other01))
    mids = 0.5 * (top_z[None, :] + other_z)

    reps = int(np.ceil(n / mids.shape[0]))
    base = np.vstack([mids for _ in range(reps)])[:n]

    eig = pca.explained_variance_
    rel = np.sqrt(eig / np.max(eig))
    scales = jitter * rel

    Z = base + rng.normal(0.0, scales, size=base.shape)
    X01 = xs.inverse_transform(pca.inverse_transform(Z))
    return np.clip(X01, 0.0, 1.0)

def pick_kmeans_in_pc(Z, seed):
    best = None
    for k in range(2, 7):
        km = KMeans(n_clusters=k, random_state=seed, n_init=10)
        labels = km.fit_predict(Z)
        sil = silhouette_score(Z, labels)
        if (best is None) or (sil > best["sil"]):
            best = {"k": k, "labels": labels, "centroids_z": km.cluster_centers_, "sil": sil}
    return best

# -----------------------------
# 5) Main
# -----------------------------
def main():
    # Strict bounds
    X = np.clip(X_raw, 0.0, 1.0)
    d = X.shape[1]
    n = X.shape[0]

    # Best observed
    best_idx = int(np.argmax(y_raw))
    x_best = X[best_idx].copy()
    f_best = float(np.max(y_raw))

    # RL-style exploration decay (more data -> less random exploration)
    explore_eps = max(0.05, 0.35 * np.exp(-n / 25.0))

    # PCA model
    xs_pca, pca, Z = pca_model(X, RANDOM_STATE, keep_var=PCA_KEEP_VAR)

    # Clustering in PC space
    clu = pick_kmeans_in_pc(Z, RANDOM_STATE)
    labels = clu["labels"]
    centroids_z = clu["centroids_z"]
    centroids01 = np.clip(xs_pca.inverse_transform(pca.inverse_transform(centroids_z)), 0.0, 1.0)

    cluster_ids = np.unique(labels)
    top_cid = max(cluster_ids, key=lambda cid: float(np.max(y_raw[labels == cid])))
    centroid_top = centroids01[top_cid].copy()
    other_cids = [c for c in cluster_ids if c != top_cid]
    centroids_other = centroids01[other_cids] if len(other_cids) else np.empty((0, d))

    # GP surrogate (fixed hyperparameters => faster + stable late-stage behavior)
    xs = StandardScaler()
    ys = StandardScaler()
    Xs = xs.fit_transform(X)
    ys_scaled = ys.fit_transform(y_raw.reshape(-1, 1)).ravel()

    kernel = (
        C(1.0, constant_value_bounds="fixed")
        * Matern(length_scale=np.ones(d), length_scale_bounds="fixed", nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds="fixed")
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=False,
        optimizer=None,
        random_state=RANDOM_STATE
    )
    gp.fit(Xs, ys_scaled)

    # Candidate pools (arms in a contextual bandit)
    Xg = sobol_global_candidates(N_GLOBAL, d, RANDOM_STATE)
    Xl_best = local_candidates_in_pc(x_best, N_LOCAL_BEST, xs_pca, pca, base_scale=0.20, seed=RANDOM_STATE + 1)
    Xl_cent = local_candidates_in_pc(centroid_top, N_LOCAL_CENTROID, xs_pca, pca, base_scale=0.28, seed=RANDOM_STATE + 2)
    Xb = boundary_candidates_in_pc(centroid_top, centroids_other, N_BOUNDARY, xs_pca, pca, jitter=0.18, seed=RANDOM_STATE + 3)

    pools = [Xg, Xl_best, Xl_cent, Xb]

    # Evaluate each pool with an optimistic EI statistic (pool "value")
    def eval_pool(Xcand):
        Xcand = min_dist_filter(Xcand, X, MIN_DIST)
        mu_s, std_s = gp.predict(xs.transform(Xcand), return_std=True)
        mu = ys.inverse_transform(mu_s.reshape(-1, 1)).ravel()
        sigma = np.maximum(std_s * float(ys.scale_[0]), 1e-12)

        # xi increases slightly when exploration is higher
        xi = 0.002 + 0.01 * explore_eps
        ei = gaussian_ei(mu, sigma, f_best, xi=xi)

        # optimistic pool value: 95th percentile EI
        pv = float(np.quantile(ei, 0.95))
        return pv, Xcand, mu, sigma, ei

    vals = []
    caches = []
    for p in pools:
        pv, Xcand, mu, sigma, ei = eval_pool(p)
        vals.append(pv)
        caches.append((Xcand, mu, sigma, ei))

    vals = np.array(vals)

    # Softmax selection over pools (bandit policy)
    temp = 0.12 + 0.35 * explore_eps
    w = np.exp((vals - vals.max()) / max(temp, 1e-9))
    w = w / w.sum()

    rng = np.random.RandomState(RANDOM_STATE + 77)
    chosen = int(rng.choice(len(pools), p=w))

    Xcand, mu, sigma, ei = caches[chosen]

    # Within-pool epsilon-greedy:
    # - explore: max uncertainty
    # - exploit: mixed EI + UCB
    if rng.rand() < explore_eps:
        idx = int(np.argmax(sigma))
    else:
        beta = 1.6 + 0.8 * explore_eps
        ucb = mu + beta * sigma
        score = 0.7 * zscore(ei) + 0.3 * zscore(ucb)
        idx = int(np.argmax(score))

    x_next = np.round(np.clip(Xcand[idx], 0.0, 1.0), 6)

    np.set_printoptions(suppress=True, formatter={"float_kind": lambda v: f"{v:.6f}"})
    print(x_next)

if __name__ == "__main__":
    main()

C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai

[0.315444 0.257588 0.430462 0.252340 0.343592 0.712713]
